In [ ]:
# ============================================================
# WEEK 7 — K-Means Clustering on PCA Components
# F1 Race Intelligence & Strategic Optimization
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples, calinski_harabasz_score, davies_bouldin_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style("darkgrid")
sns.set_palette("tab10")

print("✅ Libraries loaded successfully")

In [ ]:
# ------------------------------------------------------------
# Load PCA scores (output from PCA_V3.ipynb)
# 643 tactical events × 15 principal components
# ------------------------------------------------------------

from pathlib import Path

BASE_DIR = Path("..", "..").resolve()
PCA_PATH = BASE_DIR / "data" / "features" / "pca_scores.parquet"
MASTER_PATH = BASE_DIR / "data" / "processed" / "tactical_events" / "tactical_events_master.parquet"

df_pca = pd.read_parquet(PCA_PATH)
df_master = pd.read_parquet(MASTER_PATH)

print(f"PCA scores shape: {df_pca.shape}")
print(f"Tactical master shape: {df_master.shape}")
print(f"\nPCA columns: {list(df_pca.columns)}")
print(f"\nEvent types distribution:\n{df_pca['event_type'].value_counts()}")
print(f"\nRaces distribution:\n{df_pca['race_name'].value_counts()}")
df_pca.head(3)

In [ ]:
# ------------------------------------------------------------
# Extract the 15 PCA components as the feature matrix X
# These are already scaled (mean=0, std=1) from PCA pipeline
# ------------------------------------------------------------

PC_COLS = [f"PC{i}" for i in range(1, 16)]

X = df_pca[PC_COLS].values

print(f"Feature matrix X shape: {X.shape}")
print(f"Using components: {PC_COLS}")
print(f"\nBasic stats per component (should be ~0 mean, unit variance):")
pd.DataFrame(X, columns=PC_COLS).describe().loc[['mean','std','min','max']].round(3)

In [ ]:
# ============================================================
# PARAMETER SWEEP — K from 2 to 10
# The professor requires showing the sweep, not just one k
# ============================================================

K_RANGE = range(2, 11)
RANDOM_STATE = 42
N_INIT = 20  # multiple initializations for stability

inertias = []
silhouette_scores = []
calinski_scores = []
davies_bouldin_scores = []

print("Running K-Means sweep...")
print(f"{'k':>4} | {'Inertia':>12} | {'Silhouette':>12} | {'Calinski-H':>12} | {'Davies-B':>10}")
print("-" * 60)

for k in K_RANGE:
    km = KMeans(n_clusters=k, init='k-means++', n_init=N_INIT,
                max_iter=500, random_state=RANDOM_STATE)
    labels = km.fit_predict(X)
    
    inertia = km.inertia_
    sil = silhouette_score(X, labels)
    cal = calinski_harabasz_score(X, labels)
    dav = davies_bouldin_score(X, labels)
    
    inertias.append(inertia)
    silhouette_scores.append(sil)
    calinski_scores.append(cal)
    davies_bouldin_scores.append(dav)
    
    print(f"{k:>4} | {inertia:>12.1f} | {sil:>12.4f} | {cal:>12.2f} | {dav:>10.4f}")

print("\n✅ Sweep complete")

In [ ]:
# ------------------------------------------------------------
# Visualize all 4 metrics across the parameter sweep
# ------------------------------------------------------------

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("K-Means Parameter Sweep — k from 2 to 10\n(F1 Tactical Events on PCA Components)",
             fontsize=14, fontweight='bold', y=1.01)

k_vals = list(K_RANGE)

# 1. Elbow (Inertia)
ax1 = axes[0, 0]
ax1.plot(k_vals, inertias, 'bo-', linewidth=2, markersize=8)
ax1.axvline(x=4, color='red', linestyle='--', alpha=0.7, label='k=4 (selected)')
ax1.set_title("Elbow Method — Inertia (WCSS)", fontweight='bold')
ax1.set_xlabel("Number of Clusters (k)")
ax1.set_ylabel("Inertia (Within-Cluster Sum of Squares)")
ax1.legend()
ax1.set_xticks(k_vals)

# 2. Silhouette Score
ax2 = axes[0, 1]
ax2.plot(k_vals, silhouette_scores, 'gs-', linewidth=2, markersize=8)
ax2.axvline(x=4, color='red', linestyle='--', alpha=0.7, label='k=4 (selected)')
ax2.set_title("Silhouette Score (higher = better)", fontweight='bold')
ax2.set_xlabel("Number of Clusters (k)")
ax2.set_ylabel("Silhouette Score")
ax2.legend()
ax2.set_xticks(k_vals)

# 3. Calinski-Harabasz
ax3 = axes[1, 0]
ax3.plot(k_vals, calinski_scores, 'r^-', linewidth=2, markersize=8)
ax3.axvline(x=4, color='red', linestyle='--', alpha=0.7, label='k=4 (selected)')
ax3.set_title("Calinski-Harabasz Index (higher = better)", fontweight='bold')
ax3.set_xlabel("Number of Clusters (k)")
ax3.set_ylabel("Calinski-Harabasz Score")
ax3.legend()
ax3.set_xticks(k_vals)

# 4. Davies-Bouldin
ax4 = axes[1, 1]
ax4.plot(k_vals, davies_bouldin_scores, 'md-', linewidth=2, markersize=8)
ax4.axvline(x=4, color='red', linestyle='--', alpha=0.7, label='k=4 (selected)')
ax4.set_title("Davies-Bouldin Index (lower = better)", fontweight='bold')
ax4.set_xlabel("Number of Clusters (k)")
ax4.set_ylabel("Davies-Bouldin Score")
ax4.legend()
ax4.set_xticks(k_vals)

plt.tight_layout()
plt.savefig("../artifacts/kmeans_parameter_sweep.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: kmeans_parameter_sweep.png")